# 🎬 Talking Head Generator
**Powered by SadTalker + Starfelt monitoring**

Drop a photo + an audio file → get a video of that person saying it, with an AI GENERATED watermark burned in.

> Make sure you're on a **T4 GPU**: Runtime → Change runtime type → T4 GPU

## Step 1 — Install dependencies

In [ ]:
!pip install git+https://github.com/victorachede/starfelt.git -q
!pip install moviepy==1.0.3 -q
print('✅ Base deps installed')

## Step 2 — Clone & install SadTalker

In [ ]:
import os

if not os.path.exists('SadTalker'):
    !git clone https://github.com/OpenTalker/SadTalker.git

os.chdir('SadTalker')
!pip install -r requirements.txt -q
print('✅ SadTalker ready')

## Step 3 — Download SadTalker checkpoints

In [ ]:
import os

os.makedirs('checkpoints', exist_ok=True)
os.makedirs('gfpgan/weights', exist_ok=True)

!wget -q https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00109-model.pth.tar -O checkpoints/mapping_00109-model.pth.tar
!wget -q https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00229-model.pth.tar -O checkpoints/mapping_00229-model.pth.tar
!wget -q https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_256.safetensors -O checkpoints/SadTalker_V0.0.2_256.safetensors
!wget -q https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_512.safetensors -O checkpoints/SadTalker_V0.0.2_512.safetensors
!wget -q https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth -O gfpgan/weights/alignment_WFLW_4HG.pth
!wget -q https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth -O gfpgan/weights/detection_Resnet50_Final.pth

print('✅ Checkpoints downloaded')

## Step 4 — Upload your photo & audio file

In [ ]:
from google.colab import files
from IPython.display import display, Image

print('Upload your photo (clear, front-facing portrait works best):')
uploaded_img = files.upload()
image_path = list(uploaded_img.keys())[0]
print(f'\n✅ Got image: {image_path}')
display(Image(image_path, width=200))

In [ ]:
from google.colab import files
from IPython.display import Audio

print('Upload your audio file (.mp3 or .wav):')
uploaded_audio = files.upload()
audio_path = '/content/SadTalker/' + list(uploaded_audio.keys())[0]

print(f'\n✅ Got audio: {audio_path}')
Audio(audio_path)

## Step 5 — Run SadTalker (tracked by Starfelt)

In [ ]:
import subprocess, os, time

inference_script = f'''import subprocess, sys
result = subprocess.run([
    "python", "inference.py",
    "--driven_audio", "{audio_path}",
    "--source_image", "{image_path}",
    "--result_dir", "./results",
    "--still",
    "--preprocess", "full",
    "--enhancer", "gfpgan"
], capture_output=False)
sys.exit(result.returncode)
'''

with open('sadtalker_inference.py', 'w') as f:
    f.write(inference_script)

if not os.path.exists('starfelt.yaml'):
    !starfelt init

print('🚀 Running SadTalker via Starfelt...')
print('This takes 2-5 mins. Watch the logs below.\n')

start = time.time()
subprocess.run(['starfelt', 'run', 'sadtalker_inference.py'])
print(f'\n⏱️ Finished in {time.time() - start:.1f}s')

## Step 6 — Burn AI GENERATED watermark

In [ ]:
import glob
from moviepy.editor import VideoFileClip, TextClip, CompositeVideoClip

output_videos = glob.glob('./results/**/*.mp4', recursive=True)

if not output_videos:
    print('❌ No video found. Check logs above for errors.')
else:
    raw_video_path = sorted(output_videos)[-1]
    print(f'Found video: {raw_video_path}')

    clip = VideoFileClip(raw_video_path)

    watermark = TextClip(
        '⚠ AI GENERATED',
        fontsize=28,
        color='white',
        font='DejaVu-Sans-Bold',
        stroke_color='black',
        stroke_width=2
    ).set_duration(clip.duration)

    watermark = watermark.set_position(('center', 'bottom')).margin(bottom=12, opacity=0)
    final = CompositeVideoClip([clip, watermark])

    FINAL_PATH = '/content/ai_generated_video.mp4'
    final.write_videofile(FINAL_PATH, codec='libx264', audio_codec='aac', verbose=False, logger=None)
    print(f'✅ Watermarked video saved: {FINAL_PATH}')

## Step 7 — Preview & download

In [ ]:
from IPython.display import HTML
HTML(f'<video width="400" controls><source src="{FINAL_PATH}" type="video/mp4"></video>')

In [ ]:
from google.colab import files
files.download(FINAL_PATH)
print('✅ Downloading...')

## Step 8 — Starfelt run stats

In [ ]:
!starfelt status
print('\n--- Cost history ---')
!starfelt cost

In [ ]:
import subprocess, re
status_out = subprocess.run(['starfelt', 'status'], capture_output=True, text=True).stdout
run_ids = re.findall(r'run_[a-z0-9]+', status_out)
if run_ids:
    !starfelt inspect {run_ids[-1]}
else:
    print('Run: starfelt status to find your run ID')